We'll continue building classifiers for the Hillenbrand data, this time by using a fully-connected `Linear` layer, implemented as a custom PyTorch [nn.Module](https://docs.pytorch.org/docs/2.15/generated/torch.nn.Module.html), to implement logistic regression.

We'll compare the result to the `sklearn` model from before moving on to CNNs and ResNet34.

**Note**: [Review](https://apxml.com/courses/cnns-for-computer-vision).

In [257]:
import einops
import torch as t
import torch.nn as nn

In [258]:
device = t.device('mps' if t.backends.mps.is_available() else 'cuda' if t.cuda.is_available() else 'cpu')

## Neural Nets & Logistic Regression

TODO

- Review the architecture of a basic neural net
- Show equivalence via tensor shapes etc. to logistic regression model
- Implement custom `Linear`; custom `Softmax`; define architecture
  - Discuss `weight_decay` vs `C`, etc.
- Training loop
- Inspection
  - Compare weights to logistic regression feature vectors
- Interpretability compared to bootstrapping ish above

## Multi-Layer Perceptrons

TODO

- MLP architecture, characteristics

TODO

- Implement a few more custom layers
- Build MLP architecture
- Learn how to use DataLoaders etc. against Fashion MNIST
- Training loop with validation, etc.

In [259]:
class ReLU(nn.Module):
    def forward(x: t.Tensor) -> t.Tensor:
        return t.max(t.zeros_like(x), x)

In [260]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features, bias = True):
        super.__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.bias = bias if bias else None

        # kaiman init
        sf = 1 / np.sqrt(self.in_features)
        weight = sf * (2 * t.rand(out_features, in_features) - 1)
        self.weight = nn.Parameter(weight)

        if self.bias:
            bias_ = sf * (2 * t.rand(out_features) - 1)
            self.bias = nn.Parameter(bias_)

    def forward(self, x: t.Tensor):
        x = einops.einsum(x, self.weight, '... in_features, out_features in_features -> ... out_features')
        if self.bias:
            x += self.bias
        return x

In [261]:
# from ARENA: https://colab.research.google.com/github/ARENA-education/ARENA_materials/blob/main/chapter0_fundamentals/exercises/part2_cnns/0.2_CNNs_&_ResNets_exercises.ipynb?t=20261018#scrollTo=ABoAq-qCAa9W
class Flatten(nn.Module):
    def __init__(self, start_dim: int = 1, end_dim: int = -1) -> None:
        super().__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim

    def forward(self, input: t.Tensor) -> t.Tensor:
        """
        Flatten out dimensions from start_dim to end_dim, inclusive of both.
        """
        shape = input.shape

        # Get start & end dims, handling negative indexing for end dim
        start_dim = self.start_dim
        end_dim = self.end_dim if self.end_dim >= 0 else len(shape) + self.end_dim

        # Get the shapes to the left / right of flattened dims, as well as size of flattened middle
        shape_left = shape[:start_dim]
        shape_right = shape[end_dim + 1 :]
        shape_middle = t.prod(t.tensor(shape[start_dim : end_dim + 1])).item()

        return t.reshape(input, shape_left + (shape_middle,) + shape_right)

    def extra_repr(self) -> str:
        return ", ".join([f"{key}={getattr(self, key)}" for key in ["start_dim", "end_dim"]])

In [264]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = Flatten()
        self.relu = ReLU()
        self.linear1 = Linear(in_features=28 * 28, out_features=100)
        self.linear2 = Linear(in_features=100, out_features=10)

    def forward(self, x: t.Tensor) -> t.Tensor:
        return self.linear2(self.relu(self.linear1(self.flatten(x))))

TODO

- We still need to figure out how to load data
- Let's just use MNIST so we have something to run on, get that sorted, then proceed to loading the mel spectrogram data and training the MLP on it
- Won't be a good result but will confirm we're ready to use it to train new architectures

TODO

- Explain DataLoaders, transforms, etc. (briefly, only what's needed for Mel later)

In [ ]:
mnist_trainset, _ = get_mnist()
mnist_trainloader = DataLoader(mnist_trainset, batch_size=batch_size, shuffle=True)

TODO

- Review training process: batches over epochs, optimization/gradient descent, loss functions, progress tracking (ergonomic but mention `tqdm`)

In [ ]:
batch_size = 128
epochs = 3

In [ ]:
model = MLP().to(device)

In [ ]:
optimizer = t.optim.Adam(model.parameters(), lr=1e-3)
loss_list = []

for epoch in range(epochs):
    pbar = tqdm(mnist_trainloader)

    for imgs, labels in pbar:
        # Move data to device, perform forward pass
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)

        # Calculate loss, perform backward pass
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Update logs & progress bar
        loss_list.append(loss.item())
        pbar.set_postfix(epoch=f"{epoch + 1}/{epochs}", loss=f"{loss:.3f}", refresh=False)

TODO

- Evaluate results, but again, this is just for shape

## Interlude: Preparing Mel Spectrograms for Deep Learning

TODO

- Review representing images as tensors
- Create dataset from Mel spectrograms
- Create better `Trainer` harness and include better logging, etc., so that next time we can just swap the architecture and such (we'll need a couple CNN architectures and later a ResNet)

In [ ]:
# TODO

## CNNs: Convolution, Pooling, & Receptive Fields

TODO

- Review CNN architecture, pooling, receptive fields; mostly link out to existing material and just focus on what's necessary for the Torch work and to support the math epilogue

In [ ]:
# TODO: layers, arch, and shit; define a couple of CNN architectures

## CNNs on Hillenbrand Spectrogram Data

TODO

- Basically, just train the CNN on the spectrogram data
- Run through a proper evaluation/interpretation run: i.e., run the steps a data scientist would in industry to evaluate and characterize
  - Wrap in a tool for reuse, see below

## ResNet34  

TODO

- Architecture, new layers, implementation
- Same as above; train, evaluate, etc.
- ...But maybe use Weights & Biases this time

TODO: Conclusions/Summary

- Discuss "results"; again, mostly technique demos, not results

## Interlude: Convolution, DFT, and Geometric Invariants

TODO: Convolution Theorem + shift invariance as a consequence of mutually commuting operators sharing eigenvectors; generalizes to infinite dimensions and via block diagonalization

This will be a separate, third entry; a relatively short one, no proofs, just notes and such.

## References


1. J. Hillenbrand, L. A. Getty, M. J. Clark, and K. Wheeler, "Acoustic characteristics of American English vowels," J. Acoust. Soc. Am., vol. 97, no. 5, pt. 1, pp. 3099–3111, May 1995, doi: 10.1121/1.411872.
2. G. E. Peterson and H. L. Barney, "Control methods used in a study of the vowels," J. Acoust. Soc. Am., vol. 24, no. 2, pp. 175–184, Mar. 1952, doi: 10.1121/1.1906875.
3. P. Ladefoged and K. Johnson, A Course in Phonetics, 6th ed. Boston, MA: Wadsworth, 2011.
4. B. H. Story and K. Bunton, "Formant measurement in children's speech based on spectral filtering," Speech Commun., vol. 76, pp. 93–111, Feb. 2016, doi: 10.1016/j.specom.2015.11.001.
5. P. Ladefoged and D. E. Broadbent, “Information conveyed by vowels,” The Journal of the Acoustical Society of America, vol. 29, no. 1, pp. 98–104, Jan. 1957. doi:10.1121/1.1908694 
6. Barlaz, M. (n.d.). Vowel Normalization in R. Marissa Barlaz, PhD. https://marissabarlaz.github.io/portfolio/vowelnormalization/#what-is-vowel-normalization
7. Harris, F. J., "On the use of windows for harmonic analysis with the discrete Fourier transform," *Proc. IEEE*, vol. 66, no. 1, pp. 51–83, 1978.
8. P. Adank, R. van Hout, and R. Smits, "An acoustic description of the vowels of Northern and Southern Standard Dutch," J. Acoust. Soc. Amer., vol. 116, no. 3, pp. 1729–1738, Sep. 2004, doi: 10.1121/1.1779271
9. https://www.isca-archive.org/interspeech_2021/heeringa21_interspeech.pdf
10. https://pubmed.ncbi.nlm.nih.gov/32538265/
11. https://arxiv.org/pdf/1906.02590
